In [1]:
import geemap
import ee

Map = geemap.Map()

In [2]:
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive

gauth = GoogleAuth()
gauth.LocalWebserverAuth()
drive = GoogleDrive(gauth)

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?client_id=1054638670672-9u1bi43kofatmvbrqhu2663nch52nvs0.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8090%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive&access_type=online&response_type=code

Authentication successful.


In [3]:
roi = geemap.geojson_to_ee(r'E:/Data/GIS/JX.json')
modis = ee.ImageCollection("MODIS/006/MOD09Q1").filterDate('2016', '2017')
landsat8 = ee.ImageCollection('LANDSAT/LC08/C01/T1_8DAY_NDVI')\
            .filterDate('2016', '2017')
modis_names = modis.aggregate_array('system:id').getInfo()
landsat_names = landsat8.aggregate_array('system:id').getInfo()

In [4]:
def clip(img):
    return img.clip(roi.geometry())
def modis_NDVI(img):
    img1 = img.clip(roi.geometry())
    img2 = img1.normalizedDifference(['sur_refl_b02', 'sur_refl_b01'])
    return img2
def export_image(i):
    modis_name = modis_names[i]
    landsat_name = landsat_names[i]
    modis_img = modis_NDVI(ee.Image(modis_name))
    landsat_img = clip(ee.Image(landsat_name))
    NDVI_img = landsat_img.unmask(modis_img)
    f_name = 'NDVI_'+modis_name.split('/')[-1]
    geemap.ee_export_image_to_drive(NDVI_img, description=f_name,
                                    folder='Image', scale=30)
    return f_name+'.tif'

In [13]:
t = 'NDVI_2016_02_26.tif'
file_list = drive.ListFile({'q':"title='{0}'".format(t)}).GetList()
out_file = path + t
file = file_list[0]
file.GetContentFile(out_file)
print(out_file+'下载成功！！')
file.Delete()

E:/Data/NDVI_JX/NDVI_2016_02_26.tif下载成功！！


In [ ]:
import time
path = r'E:/Data/NDVI_JX/'
# title = export_image(6)
for i in range(9, len(modis_names)+1):
    out_file = path + title
    file_list = []
    while(len(file_list) == 0):
        file_list = drive.ListFile({'q':"title='{0}'".format(title)}).GetList()
    if i != len(modis_names):
        title = export_image(i)
    file = file_list[0]
    file.GetContentFile(out_file)
    print(out_file+'下载成功！！')
    file.Delete()

Exporting NDVI_2016_03_13 ...
E:/Data/NDVI_JX/NDVI_2016_03_05.tif下载成功！！
Exporting NDVI_2016_03_21 ...


In [17]:
title

'NDVI_2016_03_05.tif'

In [16]:
modis_names[8]

'MODIS/006/MOD09Q1/2016_03_05'

In [10]:
title

'NDVI_2016_03_05.tif'